# Sandman Version 52 Current-Only Candidate

Single-run Polymer Property Prediction Round 2 pipeline. The notebook discovers the
official competition input bundle, performs EDA, rebuilds descriptors and target
models from scratch, assembles the fixed target-specific compound route, validates
the output schema, and writes `Sandman_Version_52_8th_Aug_without_archive.csv`.

This current-only candidate uses only official current `train.csv` and official `test.csv`; the archive label file is not loaded by the active route. No non-official runtime inputs are used.


In [ ]:
from pathlib import Path
import hashlib, json, platform
import numpy as np
import pandas as pd

SEED = 20260809
TARGETS = ['tg', 'egc', 'egb', 'ei', 'eea', 'nc', 'eps']
np.random.seed(SEED)

def sha256_file(path):
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for block in iter(lambda: handle.read(1 << 20), b''):
            digest.update(block)
    return digest.hexdigest()

def locate_bundle():
    here = Path.cwd().resolve()
    candidates = []
    for parent in (here, *here.parents):
        candidates.append(parent / 'ppp-round-2')
    candidates.extend([
        Path('/kaggle/input/ppp-round-2'),
        Path('/kaggle/input/polymer-property-prediction-round-2/ppp-round-2'),
        Path('/kaggle/input/aisehack-2-0-polymer-property-prediction-round-2/ppp-round-2'),
    ])
    for candidate in candidates:
        if (candidate / 'train.csv').is_file() and (candidate / 'test.csv').is_file():
            return candidate.resolve()
    raise FileNotFoundError('official ppp-round-2 input bundle was not found')

DATA_DIR = locate_bundle()
train = pd.read_csv(DATA_DIR / 'train.csv')
test = pd.read_csv(DATA_DIR / 'test.csv')
print(json.dumps({
    'python': platform.python_version(),
    'data_dir': str(DATA_DIR),
    'train_rows': int(len(train)),
    'test_rows': int(len(test)),
    'train_sha256': sha256_file(DATA_DIR / 'train.csv'),
    'test_sha256': sha256_file(DATA_DIR / 'test.csv'),
}, sort_keys=True))
print('target counts train:')
display(train.groupby('target_type').target.agg(['count','mean','std','min','max']).round(6))
print('target counts test:')
display(test.target_type.value_counts().sort_index().to_frame('rows'))
assert list(train.columns) == ['smiles', 'target', 'target_type']
assert list(test.columns) == ['id', 'smiles', 'target_type']
assert len(test) == 4940
assert test['id'].is_unique
assert np.array_equal(test['id'].to_numpy(int), np.arange(1, 4941))


## Architecture and route

Pipeline class: current-only fixed target-route compound.

The runtime source bundle implements:

- canonical SMILES normalization and duplicate/label-availability checks;
- RDKit descriptor blocks, Morgan/atom-pair/topological-torsion/path fingerprints,
  MACCS keys, polymer repeat-unit topology, oligomer-style descriptors, and character
  n-gram features;
- target-specific ridge, tree-ensemble, local-similarity, low-rank, Polymer
  Genome-style, electronic/ionic, and cross-property carriers;
- fixed per-target routing and signed residual blends selected before this notebook
  run; and
- final `id,target` CSV validation in official test order.

Estimated execution time is branch-dependent: about 30-90 minutes for the archive
notebooks and 20-60 minutes for the current-only notebooks on the development
workstation. No network access is required after Python dependencies are present.


In [ ]:
import base64, io, os, tarfile, subprocess, sys

BUNDLE_B64 = '...'
BUNDLE_ROOT = Path.cwd() / '.sandman_source_bundle' / 'noarchive_rank1'
BUNDLE_ROOT.mkdir(parents=True, exist_ok=True)
with tarfile.open(fileobj=io.BytesIO(base64.b64decode(BUNDLE_B64)), mode='r:gz') as tar:
    tar.extractall(BUNDLE_ROOT)

env = os.environ.copy()
env['SANDMAN_VARIANT_KEY'] = 'noarchive_rank1'
env['SANDMAN_OUTPUT_CSV'] = str(Path.cwd() / 'Sandman_Version_52_8th_Aug_without_archive.csv')
env['SANDMAN_BUNDLE_ROOT'] = str(BUNDLE_ROOT)
completed = subprocess.run(
    [sys.executable, str(BUNDLE_ROOT / 'scripts' / 'sandman_runner.py')],
    cwd=Path.cwd(),
    env=env,
    text=True,
    stdout=subprocess.PIPE,
    stderr=subprocess.STDOUT,
)
print(completed.stdout)
if completed.returncode != 0:
    raise RuntimeError('Sandman generation failed')


In [ ]:
OUTPUT = Path.cwd() / 'Sandman_Version_52_8th_Aug_without_archive.csv'
submission = pd.read_csv(OUTPUT)
assert list(submission.columns) == ['id', 'target']
assert len(submission) == 4940
assert submission['id'].is_unique
assert np.array_equal(submission['id'].to_numpy(int), np.arange(1, 4941))
assert np.isfinite(submission['target'].to_numpy(float)).all()
print(json.dumps({
    'output': str(OUTPUT),
    'rows': int(len(submission)),
    'sha256': sha256_file(OUTPUT),
    'target_min': float(submission['target'].min()),
    'target_max': float(submission['target'].max()),
}, sort_keys=True))
display(submission.head())
